# Goodreads Predictions Explorer

This notebook explores the output of the Goodreads Ranker pipeline stored in the SQLite database (`data/goodreads.db`).

It reviews the ratings, recommendations, and predictions generated by the ensemble models (`solo_pred_rating`, `friend_pred_rating`, `pred_rating`, and `final_rating`).

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

In [ ]:
db_path = os.path.join("data", "goodreads.db")
print(f"Connecting to database: {db_path}")
conn = sqlite3.connect(db_path)

# Query to load predictions joined with book metadata and user's read/rating status
# We join reader_libraries via editions_lookup to ensure that if you read/rated any edition 
# of a book, it is mapped to the canonical bestBook record correctly.
query = """
SELECT 
    bp.book_id,
    b.title,
    b.description,
    b.num_pages,
    b.language_name,
    b.web_url,
    b.publisher,
    bp.solo_pred_rating,
    bp.friend_pred_rating,
    bp.count_adjusted_rating,
    bp.pred_rating,
    bp.final_rating,
    bp.date_updated,
    MAX(rl.rating) as my_rating,
    MAX(CASE WHEN rl.book_id IS NOT NULL THEN 1 ELSE 0 END) as in_library,
    MAX(CASE WHEN rl.rating IS NOT NULL AND rl.rating > 0 THEN 1 ELSE 0 END) as is_rated_by_me,
    (b.star_1 + b.star_2 + b.star_3 + b.star_4 + b.star_5) as ratings_count,
    (b.star_1 * 1.0 + b.star_2 * 2.0 + b.star_3 * 3.0 + b.star_4 * 4.0 + b.star_5 * 5.0) / 
        NULLIF(b.star_1 + b.star_2 + b.star_3 + b.star_4 + b.star_5, 0) as avg_rating
FROM book_predictions bp
JOIN books b ON bp.book_id = b.legacy_id
LEFT JOIN editions_lookup el ON bp.book_id = el.canonical_book_id
LEFT JOIN reader_libraries rl ON el.raw_legacy_id = rl.book_id AND rl.list_id = (SELECT list_id FROM readers WHERE is_self = 1)
GROUP BY bp.book_id
"""
df = pd.read_sql_query(query, conn)
print(f"Loaded {len(df)} predictions.")

## Basic Dataset Information

Let's look at the shape of the dataset, data types, and check for any missing values.

In [ ]:
df.info()

In [ ]:
df.describe()

## Library Coverage and Scraped Books Status

Let's see the proportion of books that are in your library, rated, vs new recommendations.

In [ ]:
print(f"Total books in the system: {len(df)}")
print(f"Books in your library (canon or any edition): {df['in_library'].sum()} ({df['in_library'].mean()*100:.1f}%)")
print(f"Books rated by you (training set): {df['is_rated_by_me'].sum()} ({df['is_rated_by_me'].mean()*100:.1f}%)")
print(f"New recommendation candidates: {len(df) - df['in_library'].sum()} ({(1 - df['in_library'].mean())*100:.1f}%)")

## Distribution of Prediction Scores

We can plot the histograms for each of the prediction target metrics:
- `solo_pred_rating`: Taste match score based purely on your own preferences.
- `friend_pred_rating`: Taste match score based on calibrated preferences of similar friends.
- `pred_rating`: Combining solo & friend (`solo_pred_rating * friend_pred_rating`).
- `final_rating`: Combined rating scaled by public rating popularity (`pred_rating * count_adjusted_rating`).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['solo_pred_rating'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Solo Predicted Rating Distribution')
axes[0, 0].set_xlabel('Score')

sns.histplot(df['friend_pred_rating'], kde=True, ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Friend Predicted Rating Distribution')
axes[0, 1].set_xlabel('Score')

sns.histplot(df['pred_rating'], kde=True, ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Combined Prediction Rating (Solo * Friend)')
axes[1, 0].set_xlabel('Score')

sns.histplot(df['final_rating'], kde=True, ax=axes[1, 1], color='violet')
axes[1, 1].set_title('Final Recommendation Rating (Combined * Popularity Adjusted)')
axes[1, 1].set_xlabel('Score')

plt.tight_layout()
plt.show()

## Correlation Analysis

Let's see how our prediction scores correlate with public Goodreads ratings (`avg_rating`) and your actual ratings (`my_rating`).

In [ ]:
corr_cols = [
    'solo_pred_rating', 
    'friend_pred_rating', 
    'count_adjusted_rating', 
    'pred_rating', 
    'final_rating', 
    'avg_rating', 
    'my_rating'
]
corr_df = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, cmap='coolwarm', fmt=".3f", vmin=-1, vmax=1)
plt.title('Correlation Matrix of Predictions and Ratings')
plt.show()

## Comparing Personal (Solo) Taste vs. Friend-Based Taste

Let's plot a scatter plot of `solo_pred_rating` vs. `friend_pred_rating` to visualize where your taste and your friends' tastes align.

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=df, 
    x='solo_pred_rating', 
    y='friend_pred_rating', 
    hue='is_rated_by_me', 
    palette={0: 'gray', 1: 'blue'}, 
    alpha=0.6
)
plt.title('Solo vs. Friend Predicted Ratings')
plt.xlabel('Solo Predicted Rating')
plt.ylabel('Friend Predicted Rating')
plt.legend(title='Rated by Me')
plt.show()

## Top Recommendations (Not in Library)

Now let's filter out books already in your library and review the top recommendations generated by the pipeline.

In [ ]:
# Filter to books not currently in library
recommendations = df[df['in_library'] == 0].copy()

cols_to_show = ['title', 'final_rating', 'pred_rating', 'solo_pred_rating', 'friend_pred_rating', 'avg_rating', 'ratings_count']

### 1. Top 15 Recommendations by `final_rating`

In [ ]:
print("Top 15 Books by Final Recommendation Rating (Adjusted for Ratings Count):")
recommendations.sort_values(by='final_rating', ascending=False)[cols_to_show].head(15)

### 2. Top 15 Recommendations by `pred_rating` (Taste only, no size/popularity bias)

In [ ]:
print("Top 15 Books by Combined Taste Match (Solo * Friend):")
recommendations.sort_values(by='pred_rating', ascending=False)[cols_to_show].head(15)

### 3. Top 15 Recommendations by `solo_pred_rating` (Purely your taste)

In [ ]:
print("Top 15 Books by Solo predicted preference:")
recommendations.sort_values(by='solo_pred_rating', ascending=False)[cols_to_show].head(15)

### 4. Top 15 Recommendations by `friend_pred_rating` (Purely friend alignment)

In [ ]:
print("Top 15 Books by Friend predicted preference:")
recommendations.sort_values(by='friend_pred_rating', ascending=False)[cols_to_show].head(15)

## Visualizing Top Recommendations and their Characteristics

Let's see a scatter plot of the top 50 books sorted by `final_rating` mapping their ratings count against avg rating, sized by `final_rating`.

In [ ]:
top_50 = recommendations.sort_values(by='final_rating', ascending=False).head(50)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    x=top_50['avg_rating'], 
    y=top_50['ratings_count'],
    s=top_50['final_rating'] * 400, # Size represents recommendation rating
    c=top_50['solo_pred_rating'], 
    cmap='viridis', 
    alpha=0.7,
    edgecolors='black'
)

plt.title('Top 50 Recommendations: Popularity vs. Public Rating')
plt.xlabel('Public Average Rating')
plt.ylabel('Public Ratings Count (Log Scale)')
plt.yscale('log')
cbar = plt.colorbar(scatter)
cbar.set_label('Solo Predicted Rating')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()

## Clean up

Close connection.

In [ ]:
conn.close()
print("Database connection closed.")

In [ ]:
df['rating'] = df['final_rating'] * df['solo_pred_rating']

df[df['is_rated_by_me']!= 1].sort_values(by='rating', ascending=False)